In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
import argparse
import os
import sys

In [14]:
CLASS_NAMES = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew',
    'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy',
    'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy',
    'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot',
    'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus', 'Tomato___healthy'
]

In [15]:
# ── Config ───────────────────────────────────────────────────
MODEL_PATH  = 'mobilenet_plantvillage.pth'
NUM_CLASSES = 38
IMG_SIZE    = 224
device      = torch.device("cpu")

In [16]:
def load_model():
    model = models.mobilenet_v2(weights=None)
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.last_channel, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, NUM_CLASSES)
    )
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    return model

In [17]:
def preprocess(image_path):
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    return transform(img).unsqueeze(0)

In [18]:
def predict(image_path, model):
    if not os.path.exists(image_path):
        print(f"Error: File not found → {image_path}")
        return
 
    tensor = preprocess(image_path).to(device)
 
    with torch.no_grad():
        output = model(tensor)
        probs  = F.softmax(output, dim=1)
 
    top5_probs, top5_idx = torch.topk(probs, 5)
 
    print("\n" + "="*60)
    print(f"  IMAGE      : {image_path}")
    print("="*60)
    print(f"  PREDICTION : {CLASS_NAMES[top5_idx[0][0].item()]}")
    print(f"  CONFIDENCE : {top5_probs[0][0].item() * 100:.2f}%")
    print("-"*60)
    print("  TOP 5 PREDICTIONS:")
    for i in range(5):
        cls  = CLASS_NAMES[top5_idx[0][i].item()]
        prob = top5_probs[0][i].item() * 100
        print(f"    {i+1}. {cls:<50} {prob:.2f}%")
    print("="*60 + "\n")
 
    pred_class = CLASS_NAMES[top5_idx[0][0].item()]
    if 'healthy' in pred_class.lower():
        print("  STATUS  : ✅ Plant appears HEALTHY")
    else:
        print("  STATUS  : ⚠️  Disease DETECTED — consult an agronomist")
    print()

In [20]:
if __name__ == '__main__':
    if 'ipykernel' in sys.modules:
        # ✅ Running inside Jupyter Notebook
        # Change this path to any leaf image from your dataset
        image_path = r'C:\Users\Lenovo\Desktop\color\Apple___Black_rot\0bc40cc3-6a85-480e-a22f-967a866a56a1___JR_FrgE.S 2784.JPG'   # ← CHANGE THIS LINE
 
        print("\nLoading model...")
        model = load_model()
        print("Model ready ✅")
        predict(image_path, model)
 
    else:
        # ✅ Running in Terminal
        # Usage: python week4_inference.py --image path/to/leaf.jpg
        parser = argparse.ArgumentParser(description='Plant Disease Inference')
        parser.add_argument('--image', type=str, required=True,
                            help='Path to the leaf image (jpg/png)')
        args = parser.parse_args()
 
        print("\nLoading model...")
        model = load_model()
        print("Model ready ✅")
        predict(args.image, model)


Loading model...
Model ready ✅

  IMAGE      : C:\Users\Lenovo\Desktop\color\Apple___Black_rot\0bc40cc3-6a85-480e-a22f-967a866a56a1___JR_FrgE.S 2784.JPG
  PREDICTION : Apple___Black_rot
  CONFIDENCE : 98.68%
------------------------------------------------------------
  TOP 5 PREDICTIONS:
    1. Apple___Black_rot                                  98.68%
    2. Apple___Cedar_apple_rust                           1.09%
    3. Peach___Bacterial_spot                             0.07%
    4. Corn_(maize)___Common_rust_                        0.06%
    5. Pepper,_bell___Bacterial_spot                      0.05%

  STATUS  : ⚠️  Disease DETECTED — consult an agronomist

